<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W7D2_Vector_Databases_RAG_Frean_Debohi_Grace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 – Day 2: Vector Databases and RAG Chatbots

**Étudiante : Frean Debohi Grace**  
**Bootcamp : COT GenAI & Machine Learning – 2026**

## Objectifs

Ce notebook réalise entièrement les cinq exercices demandés :

1. chargement et préparation du dataset `labelled_newscatcher_dataset.csv` ;
2. création d'embeddings avec Sentence Transformers ;
3. indexation et recherche sémantique avec FAISS ;
4. stockage et interrogation avec ChromaDB ;
5. création d'un mini-pipeline RAG avec un modèle causal Hugging Face.

> Le notebook utilise seulement un sous-ensemble de 1 000 articles afin de rester rapide sur Google Colab. Le modèle génératif est volontairement compact pour fonctionner avec les ressources gratuites de Colab.

In [ ]:
# Installation des bibliothèques nécessaires.
# Exécuter cette cellule une seule fois au début du notebook.

%pip install -q -U faiss-cpu chromadb sentence-transformers transformers accelerate kagglehub

In [ ]:
# EXERCICE 1 — Chargement et préparation des données

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import faiss
import chromadb

from sentence_transformers import SentenceTransformer, InputExample
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display

# Source publique du dataset. Un téléchargement Kaggle sert de solution de secours.
DATA_URL = (
    "https://s3.wasabisys.com/iguazio/data/genai-tutorial/"
    "labelled_newscatcher_dataset.csv"
)

try:
    pdf = pd.read_csv(DATA_URL, sep=";", on_bad_lines="skip")
    dataset_source = DATA_URL
except Exception as error:
    print("Téléchargement direct indisponible, utilisation de KaggleHub :", error)
    import kagglehub

    dataset_dir = Path(
        kagglehub.dataset_download("kotartemiy/topic-labeled-news-dataset")
    )
    csv_files = list(dataset_dir.rglob("labelled_newscatcher_dataset.csv"))

    if not csv_files:
        raise FileNotFoundError(
            "Le fichier labelled_newscatcher_dataset.csv est introuvable."
        )

    dataset_source = str(csv_files[0])
    pdf = pd.read_csv(csv_files[0], sep=";", on_bad_lines="skip")

# Nettoyage minimal et création d'un identifiant unique.
required_columns = {"title", "topic"}
missing_columns = required_columns.difference(pdf.columns)
if missing_columns:
    raise ValueError(f"Colonnes obligatoires manquantes : {missing_columns}")

pdf = pdf.dropna(subset=["title", "topic"]).copy()
pdf["title"] = pdf["title"].astype(str).str.strip()
pdf["topic"] = pdf["topic"].astype(str).str.strip()
pdf = pdf[pdf["title"].ne("")].reset_index(drop=True)
pdf["id"] = np.arange(len(pdf), dtype=np.int64)

# Sous-ensemble demandé pour accélérer les expérimentations.
pdf_subset = pdf.head(1000).copy()

print(f"Source utilisée : {dataset_source}")
print(f"Dimensions du dataset complet : {pdf.shape}")
print(f"Dimensions du sous-ensemble : {pdf_subset.shape}")
print("\nValeurs manquantes du sous-ensemble :")
display(pdf_subset.isna().sum().to_frame("valeurs_manquantes"))
display(pdf_subset.head())

In [ ]:
# EXERCICE 2 — Vectorisation avec Sentence Transformers

def example_create_fn(doc1) -> InputExample:
    """Transforme un titre en objet InputExample."""
    text = str(doc1)
    return InputExample(guid=text[:80], texts=[text], label=0.0)

faiss_train_examples = pdf_subset.apply(
    lambda row: example_create_fn(row["title"]),
    axis=1,
).tolist()

print("Exemples InputExample :")
print(faiss_train_examples[:3])

# Modèle léger produisant des embeddings de 384 dimensions.
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

titles_list = pdf_subset["title"].tolist()

faiss_title_embedding = embedding_model.encode(
    titles_list,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

print("\nNombre d'embeddings :", len(faiss_title_embedding))
print("Dimension d'un embedding :", len(faiss_title_embedding[0]))
print("Forme de la matrice :", faiss_title_embedding.shape)

## Analyse des exercices 1 et 2

Le dataset contient principalement les colonnes `title`, `topic`, `link`, `domain`, `published_date` et `lang`. La colonne `id` ajoutée dans le notebook permet d'associer chaque vecteur à son article d'origine.

Un **embedding** est une représentation numérique dense d'un texte. Deux titres ayant un sens proche doivent produire des vecteurs proches dans l'espace vectoriel, même s'ils n'utilisent pas exactement les mêmes mots.

Le modèle `all-MiniLM-L6-v2` transforme chaque titre en un vecteur de **384 valeurs**. Le sous-ensemble de 1 000 titres produit donc normalement une matrice de forme `(1000, 384)`.

L'objet `InputExample` montre comment Sentence Transformers structure les données pour un entraînement ou une évaluation. Pour la simple génération d'embeddings, la méthode `encode()` peut directement recevoir la liste des titres.

In [ ]:
# EXERCICE 3 — Indexation et recherche avec FAISS

pdf_to_index = pdf_subset.copy()
id_index = pdf_to_index["id"].to_numpy(dtype=np.int64)

# Copie normalisée pour utiliser le produit scalaire comme similarité cosinus.
content_encoded_normalized = np.asarray(
    faiss_title_embedding,
    dtype=np.float32,
).copy()

faiss.normalize_L2(content_encoded_normalized)

embedding_dimension = content_encoded_normalized.shape[1]
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(embedding_dimension))
index_content.add_with_ids(content_encoded_normalized, id_index)

print("Nombre de vecteurs indexés dans FAISS :", index_content.ntotal)

def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """Retourne les k articles les plus proches sémantiquement d'une requête."""
    if not isinstance(query, str) or not query.strip():
        raise ValueError("La requête doit être une chaîne non vide.")

    k = max(1, min(k, index_content.ntotal))

    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
    ).astype("float32")

    faiss.normalize_L2(query_vector)

    similarities, ids = index_content.search(query_vector, k)

    valid_mask = ids[0] != -1
    matched_ids = ids[0][valid_mask]
    matched_scores = similarities[0][valid_mask]

    indexed_df = pdf_to_index.set_index("id")
    results_df = indexed_df.loc[matched_ids].reset_index()
    results_df["similarities"] = matched_scores

    return results_df[
        ["id", "title", "topic", "similarities"]
    ].reset_index(drop=True)

display(search_content("animal", pdf_to_index, k=5))

In [ ]:
# EXERCICE 4 — Collection et recherche avec ChromaDB

from chromadb.utils import embedding_functions

chroma_client = chromadb.Client()
collection_name = "my_news"

# Supprime l'ancienne collection si la cellule est réexécutée.
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

chroma_embedding_function = (
    embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=embedding_model_name
    )
)

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=chroma_embedding_function,
    metadata={"hnsw:space": "cosine"},
)

chroma_subset = pdf_subset.head(100).copy()

collection.add(
    documents=chroma_subset["title"].tolist(),
    metadatas=[
        {"topic": str(topic)}
        for topic in chroma_subset["topic"].tolist()
    ],
    ids=chroma_subset["id"].astype(str).tolist(),
)

print("Nombre de documents dans ChromaDB :", collection.count())

results = collection.query(
    query_texts=["space"],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

print(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
# EXERCICE 5 — Question Answering avec un modèle Hugging Face

# Modèle causal compact et instruction-tuned.
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

text_generator = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
)

question = "What developments related to space are mentioned in the news dataset?"

# La question sert également de requête de recherche dans ChromaDB.
rag_results = collection.query(
    query_texts=[question],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

retrieved_documents = rag_results["documents"][0]
context = "\n".join(
    f"{index + 1}. {document}"
    for index, document in enumerate(retrieved_documents)
)

prompt_template = f"""
Use only the context below to answer the question.
If the context is insufficient, clearly say so.
Do not invent dates, events, or facts.

Relevant context:
{context}

User question:
{question}

Give a concise answer:
""".strip()

messages = [
    {
        "role": "system",
        "content": (
            "You are a careful question-answering assistant. "
            "Your answers must remain grounded in the retrieved context."
        ),
    },
    {"role": "user", "content": prompt_template},
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

lm_response = text_generator(
    formatted_prompt,
    max_new_tokens=200,
    do_sample=False,
    return_full_text=False,
)

answer = lm_response[0]["generated_text"].strip()

print("QUESTION\n", question)
print("\nCONTEXTE RÉCUPÉRÉ\n", context)
print("\nRÉPONSE DU MODÈLE\n", answer)

## Interprétation et conclusion

### FAISS

FAISS est une bibliothèque de recherche vectorielle légère et très rapide. Dans ce notebook, les vecteurs sont normalisés avec `faiss.normalize_L2`, puis `IndexFlatIP` calcule leur produit scalaire. Pour des vecteurs normalisés, ce produit scalaire correspond à la **similarité cosinus** : une valeur élevée indique une forte proximité sémantique.

### ChromaDB

ChromaDB agit comme une base vectorielle de plus haut niveau. Elle stocke ensemble les documents, leurs identifiants, leurs métadonnées et leurs représentations vectorielles. Contrairement à l'index FAISS utilisé ici, elle fournit directement des opérations de collection, d'ajout et de requête.

Les `distances` retournées par ChromaDB sont des distances cosinus dans cette configuration : **plus la distance est petite, plus le document est pertinent**.

### Pipeline RAG

Le pipeline final suit quatre étapes :

1. la question est transformée en embedding ;
2. ChromaDB récupère les titres les plus proches ;
3. les titres sont réunis dans un contexte ;
4. le modèle génératif produit une réponse à partir de ce contexte.

Cette architecture réduit les réponses inventées, mais ne les élimine pas totalement. Sa qualité dépend de la pertinence des documents récupérés, de la taille du contexte et de la capacité du modèle génératif.

### Limites

Le dataset contient des titres de presse et non les articles complets. Le contexte est donc court et certaines questions ne peuvent pas recevoir une réponse détaillée. De plus, les résultats du dataset ne doivent pas être considérés comme des actualités en temps réel.

In [ ]:
# EXPÉRIMENTATION — Fonction RAG réutilisable et variation du contexte

def rag_answer(
    question: str,
    n_results: int = 5,
    max_new_tokens: int = 180,
) -> dict:
    """Recherche un contexte dans ChromaDB puis génère une réponse fondée dessus."""
    if not question.strip():
        raise ValueError("La question ne peut pas être vide.")

    n_results = max(1, min(n_results, collection.count()))

    retrieved = collection.query(
        query_texts=[question],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    documents = retrieved["documents"][0]
    context_text = "\n".join(
        f"- {document}" for document in documents
    )

    user_prompt = f"""
Answer the question using only the retrieved news titles.
If the evidence is insufficient, say that the available titles are insufficient.

Context:
{context_text}

Question:
{question}
""".strip()

    chat = [
        {
            "role": "system",
            "content": "Answer faithfully from the supplied context only.",
        },
        {"role": "user", "content": user_prompt},
    ]

    model_input = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

    generated = text_generator(
        model_input,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )[0]["generated_text"].strip()

    return {
        "question": question,
        "context_size": n_results,
        "documents": documents,
        "answer": generated,
    }

# Deux tailles de contexte pour observer leur influence.
experiment_question = "What science or technology topics appear in these news titles?"

for context_size in (3, 7):
    experiment = rag_answer(
        experiment_question,
        n_results=context_size,
        max_new_tokens=140,
    )
    print("=" * 90)
    print(f"CONTEXTE : {experiment['context_size']} documents")
    print("RÉPONSE :", experiment["answer"])